# PAWC + AIS row-by-row (query x engine)

Reshapes the per-cell smoke results (`data/<profile>/gold/smoke_pawc_ais_<profile>.parquet`)
into an **Excel workbook** so you can read the computation cell-by-cell instead of only the
per-engine means.

Each row = one (query x engine) cell. `ais_rate = ais_supported_sentences / n_sentences`;
`pawc_total` = position-weighted supported-word sum. Workbook sheets:
- **row_by_row** - every cell with its inputs + computed AIS/PAWC
- **AIS_q_x_engine** / **PAWC_q_x_engine** / **sources_cited_q_x_eng** - query x engine matrices
- **per_engine_summary** - the means (for reference)
- **queries_legend** - query_id -> query_text

Offline (reads the parquet; no API calls). 50 cells = 10 queries x 5 engines (Mistral excluded; smoke sample).

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "thesis_config.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

import thesis_config as cfg
gold = pd.read_parquet(cfg.GOLD / f"smoke_pawc_ais_{cfg.RUN_PROFILE}.parquet")
reg = pd.read_parquet(cfg.QUERY_REGISTRY)[["query_id", "query_text"]]
df = gold.merge(reg, on="query_id", how="left")
print(cfg.profile_summary())
print("cells:", len(df), "| queries:", df.query_id.nunique(), "| engines:", sorted(df.engine.unique()))

RUN PROFILE: PILOT  (N=100, K=3, root=/Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/pilot)
cells: 50 | queries: 10 | engines: ['chatgpt', 'claude', 'gemini', 'kimi', 'perplexity']


## 1. Row-by-row computation
One row per (query x engine). `ais_rate = ais_supported_sentences / n_sentences`;
`fetch_rate = n_sources_fetched_ok / n_sources_cited`. Sorted by query, then engine.

In [2]:
ENGINE_ORDER = ["chatgpt", "claude", "gemini", "perplexity", "kimi"]
df["engine"] = pd.Categorical(df["engine"], categories=ENGINE_ORDER, ordered=True)
df["ais_rate"] = (df["ais_supported_sentences"] / df["n_sentences"].replace(0, np.nan)).round(3)
df["fetch_rate"] = (df["n_sources_fetched_ok"] / df["n_sources_cited"].replace(0, np.nan)).round(3)

cols = ["query_id", "query_text", "engine", "run_index",
        "n_sentences", "ais_supported_sentences", "ais_rate",
        "n_sources_cited", "n_sources_fetched_ok", "fetch_rate",
        "pawc_total", "judge_calls", "input_tokens", "output_tokens"]
row_by_row = df.sort_values(["query_id", "engine"])[cols].reset_index(drop=True)
print("row_by_row:", row_by_row.shape)
display(row_by_row)

row_by_row: (50, 14)


,query_id,query_text,engine,run_index,n_sentences,ais_supported_sentences,ais_rate,n_sources_cited,n_sources_fetched_ok,fetch_rate,pawc_total,judge_calls,input_tokens,output_tokens
0,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,chatgpt,1,7,0,0.000,0,0,NaN,0.000,7,0,0
1,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,claude,1,20,19,0.950,5,4,0.800,583.850,20,66902,754
2,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,gemini,1,28,22,0.786,8,8,1.000,456.143,28,72444,1262
3,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,perplexity,1,5,5,1.000,8,7,0.875,216.800,5,28344,233
4,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,kimi,1,22,18,0.818,9,6,0.667,477.773,22,136706,976
5,gb_05118b5a6fcf8e43,art collaborations of 2023,chatgpt,1,10,8,0.800,8,6,0.750,139.900,10,56250,353
6,gb_05118b5a6fcf8e43,art collaborations of 2023,claude,1,30,20,0.667,8,6,0.750,487.633,30,169274,1031
7,gb_05118b5a6fcf8e43,art collaborations of 2023,gemini,1,31,16,0.516,18,17,0.944,617.161,31,514844,1750
8,gb_05118b5a6fcf8e43,art collaborations of 2023,perplexity,1,5,4,0.800,5,5,1.000,115.600,5,19504,218
9,gb_05118b5a6fcf8e43,art collaborations of 2023,kimi,1,18,12,0.667,14,12,0.857,222.667,18,218573,1516


## 2. Query x engine score matrices
The same numbers as matrices - read one row across to compare all 5 engines on a single query.

In [3]:
def pivot(metric, aggfunc="first"):
    return (df.pivot_table(index=["query_id", "query_text"], columns="engine",
                           values=metric, observed=True, aggfunc=aggfunc)
              .reindex(columns=ENGINE_ORDER))

ais_pivot = pivot("ais_rate").round(3)
pawc_pivot = pivot("pawc_total").round(1)
src_pivot = pivot("n_sources_cited")

print("AIS rate  (query x engine):"); display(ais_pivot)
print("PAWC total (query x engine):"); display(pawc_pivot)
print("Sources cited (query x engine):"); display(src_pivot)

AIS rate  (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0.000,0.950,0.786,1.000,0.818
gb_05118b5a6fcf8e43,art collaborations of 2023,0.800,0.667,0.516,0.800,0.667
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0.000,0.895,0.864,0.750,0.750
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0.000,0.800,0.781,0.800,0.857
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0.000,0.891,0.000,1.000,0.911
gb_0c0058305876a645,What medicine should I take when I get a cold?,0.688,0.839,0.513,0.750,0.750
gb_157a7eb6b5972399,my service canada account log in,0.000,0.000,0.667,1.000,0.714
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0.000,0.421,0.489,0.083,0.821
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0.000,1.000,1.000,0.500,0.500


PAWC total (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0.0,583.8,456.1,216.8,477.8
gb_05118b5a6fcf8e43,art collaborations of 2023,139.9,487.6,617.2,115.6,222.7
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0.0,326.8,516.3,126.0,589.8
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0.0,231.9,767.8,180.0,784.4
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0.0,1647.2,0.0,570.4,1993.6
gb_0c0058305876a645,What medicine should I take when I get a cold?,85.3,519.0,190.5,261.7,552.6
gb_157a7eb6b5972399,my service canada account log in,0.0,0.0,280.8,105.7,144.3
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0.0,310.2,387.9,3.2,561.5
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0.0,28.0,26.5,30.5,89.5


Sources cited (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0,5,8,8,9
gb_05118b5a6fcf8e43,art collaborations of 2023,8,8,18,5,14
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0,6,7,5,10
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0,4,17,4,15
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0,11,0,7,9
gb_0c0058305876a645,What medicine should I take when I get a cold?,2,5,8,10,6
gb_157a7eb6b5972399,my service canada account log in,2,3,10,8,5
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0,9,10,4,7
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0,1,3,7,8


## 3. Per-engine summary (means, for reference)

In [4]:
summary = (df.groupby("engine", observed=True)
           .agg(cells=("query_id", "count"),
                mean_sentences=("n_sentences", "mean"),
                mean_ais=("ais_rate", "mean"),
                mean_pawc=("pawc_total", "mean"),
                mean_sources_cited=("n_sources_cited", "mean"),
                mean_sources_ok=("n_sources_fetched_ok", "mean"))
           .round(2).reindex(ENGINE_ORDER))
display(summary)

,cells,mean_sentences,mean_ais,mean_pawc,mean_sources_cited,mean_sources_ok
engine,,,,,,
chatgpt,10,15.5,0.22,38.55,1.4,0.9
claude,10,26.8,0.73,455.63,6.1,5.0
gemini,10,28.8,0.62,354.13,10.2,8.2
perplexity,10,8.1,0.74,166.47,6.4,4.8
kimi,10,18.1,0.70,547.24,8.8,6.5


## 4. Write the Excel workbook

In [5]:
out = cfg.GOLD / f"smoke_pawc_ais_rowbyrow_{cfg.RUN_PROFILE}.xlsx"
legend = df[["query_id", "query_text"]].drop_duplicates().sort_values("query_id")
with pd.ExcelWriter(out, engine="openpyxl") as xl:
    row_by_row.to_excel(xl, sheet_name="row_by_row", index=False)
    ais_pivot.to_excel(xl, sheet_name="AIS_q_x_engine")
    pawc_pivot.to_excel(xl, sheet_name="PAWC_q_x_engine")
    src_pivot.to_excel(xl, sheet_name="sources_cited_q_x_eng")
    summary.to_excel(xl, sheet_name="per_engine_summary")
    legend.to_excel(xl, sheet_name="queries_legend", index=False)

import openpyxl
print("WROTE:", out)
print("size:", out.stat().st_size, "bytes")
print("sheets:", openpyxl.load_workbook(out).sheetnames)

WROTE: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/pilot/gold/smoke_pawc_ais_rowbyrow_pilot.xlsx
size: 15260 bytes
sheets: ['row_by_row', 'AIS_q_x_engine', 'PAWC_q_x_engine', 'sources_cited_q_x_eng', 'per_engine_summary', 'queries_legend']
